# Healthcare Scheduler Agent
CareCompanion is an AI-powered healthcare support agent designed for recently discharged patients and their caregivers. The agent helps users manage post-visit care by organizing discharge instructions, tracking medications and symptoms over time, preparing users for follow-up appointments, and generating concise summaries to support effective communication with healthcare providers.

The agent operates continuously rather than as a one-time interaction. It detects upcoming appointments, prompts users for symptom updates, adapts its guidance based on patient input, and makes scheduling and reminder decisions based on discharge instructions. By maintaining context across time and interactions, CareCompanion reduces cognitive load, prevents missed care steps, and improves patient preparedness and confidence during follow-up visits.

# **Import Statements**

In [ ]:
!pip install google-generativeai>=0.3.0 pydantic>=2.0.0 python-dateutil>=2.8.0 pypdf>=3.17.0 gradio>=4.0.0 -q
!pip install google-adk -q
import os
from google.colab import userdata
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
import psycopg2
import google.generativeai as genai
import json
from langchain_core.tools import tool
from dateutil import parser as date_parser
from datetime import datetime
from typing import Optional, List, Dict
from langchain_core.prompts import ChatPromptTemplate

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


# **Part 1: Agent Environment Setup**

Setting Up API Keys

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get("NEW_GOOGLE_API_KEY")
print("✓ API key configured successfully!")

✓ API key configured successfully!


In [ ]:
model = Gemini(model="gemini-2.5-flash")

genai.configure(api_key=os.getenv("NEW_GOOGLE_API_KEY"))
_genai_model = genai.GenerativeModel("gemini-2.5-flash")

print(f"Using model: {model.model}")

Using model: gemini-2.5-flash


In [ ]:
os.environ["DB_HOST"]=userdata.get("DB_HOST")
os.environ["DB_PORT"]=userdata.get("DB_PORT")
os.environ["DB_NAME"]=userdata.get("DB_NAME")
os.environ["DB_USER"]=userdata.get("DB_USER")
os.environ["DB_PASSWORD"]=userdata.get("DB_PASSWORD")

In [ ]:
# Load secrets
for key in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD"]:
    os.environ[key]=userdata.get(key).strip()

# Connection helper
def get_db_connection():
    return psycopg2.connect(
        host=os.environ["DB_HOST"],
        database=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        port=os.environ["DB_PORT"],
        sslmode="require",
        connect_timeout=10
    )

conn=psycopg2.connect(
    host=os.environ["DB_HOST"],
    database=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    port=os.environ["DB_PORT"],
    sslmode="require",
    connect_timeout=10
)

print("✅ Connected to Neon PostgreSQL")
conn.close()

✅ Connected to Neon PostgreSQL


In [ ]:
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public';
""")
tables = cur.fetchall()
print("Existing tables:", [t[0] for t in tables])
cur.close()
conn.close()

Existing tables: ['appointments', 'users', 'patients', 'symptoms', 'medication_logs', 'notes', 'discharge_instructions', 'symptom_logs', 'reminders']


# Google Calendar API

In [ ]:
# ===============================
# GOOGLE CALENDAR (SERVICE ACCOUNT)
# ===============================

!pip install google-api-python-client google-auth -q


from google.oauth2 import service_account
from googleapiclient.discovery import build
from datetime import timedelta
from zoneinfo import ZoneInfo

SERVICE_ACCOUNT_FILE = "service_account.json"
SCOPES = ["https://www.googleapis.com/auth/calendar"]
TIMEZONE = "America/New_York"

# 🔹 Replace with your real Gmail calendar ID
CALENDAR_ID = "37a62ccd8ff30a62e6d5a86347319c2806938d0e2c008bbe93c6ff17e12aa34b@group.calendar.google.com"

def authenticate_google_calendar():
    credentials = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )
    return build("calendar", "v3", credentials=credentials)


def sync_appointment_to_google_calendar(
    appointment_time,
    summary,
    description=None,
    duration_minutes=30
):
    service = authenticate_google_calendar()

    # Ensure datetime is timezone-aware
    if appointment_time.tzinfo is None:
      appointment_time = appointment_time.replace(tzinfo=ZoneInfo("America/New_York"))

    start_time = appointment_time.isoformat()
    end_time = (appointment_time + timedelta(minutes=duration_minutes)).isoformat()

    event = {
        "summary": summary,
        "description": description or "",
        "start": {
            "dateTime": start_time,
            "timeZone": TIMEZONE,
        },
        "end": {
            "dateTime": end_time,
            "timeZone": TIMEZONE,
        },
    }

    created_event = service.events().insert(
        calendarId=CALENDAR_ID,
        body=event
    ).execute()

    print("✅ Google Calendar event created!")
    print("🔗 Event link:", created_event.get("htmlLink"))

    return created_event.get("htmlLink")

# **Part 2: Database Helper Functions**

In [ ]:
# we can add, get, and update data in our tables

def get_connection():
  return psycopg2.connect(
      host=os.environ["DB_HOST"],
      database=os.environ["DB_NAME"],
      user=os.environ["DB_USER"],
      password=os.environ["DB_PASSWORD"],
      port=os.environ["DB_PORT"],
      sslmode="require"
  )

In [ ]:
def init_db():
    conn=get_db_connection()
    cur=conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            id SERIAL PRIMARY KEY,
            name TEXT NOT NULL,
            email TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        CREATE TABLE IF NOT EXISTS appointments (
            id SERIAL PRIMARY KEY,
            patient_id INTEGER REFERENCES patients(id) ON DELETE CASCADE,
            appointment_time TIMESTAMP NOT NULL,
            reason TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        CREATE TABLE IF NOT EXISTS symptoms (
            id SERIAL PRIMARY KEY,
            patient_id INTEGER REFERENCES patients(id) ON DELETE CASCADE,
            symptom TEXT NOT NULL,
            severity INTEGER,
            logged_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    """)

    conn.commit()
    cur.close()
    conn.close()

    print("✅ Tables initialized")

In [ ]:
init_db()

✅ Tables initialized


**User** Functions

In [ ]:
def create_user(name: str, email: str = None, phone: str = None) -> int:
    """
    Add a new user to the database.
    Returns the new user_id.
    """
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        "INSERT INTO users (name,email,phone) VALUES (%s,%s,%s) RETURNING user_id;",
        (name,email,phone)
    )
    user_id=cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    print(f"Created user: {name} (ID: {user_id})")
    return user_id


def get_user(user_id: int) -> dict:
    """Get a user by their ID"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        "SELECT user_id,name,email,phone,created_at FROM users WHERE user_id=%s;",
        (user_id,)
    )
    row=cur.fetchone()

    cur.close()
    conn.close()

    if row:
        return {
            "user_id":row[0],
            "name":row[1],
            "email":row[2],
            "phone":row[3],
            "created_at":row[4]
        }
    return None

**Appointment** Functions

In [ ]:
def create_appointment(patient_id: int, appointment_time, reason: str = None) -> int:
    """Schedule an appointment for a patient"""

    conn = get_connection()
    cur = conn.cursor()

    cur.execute(
        """
        INSERT INTO appointments (patient_id, appointment_time, reason)
        VALUES (%s, %s, %s)
        RETURNING id;
        """,
        (patient_id, appointment_time, reason)
    )

    appointment_id = cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    # ----------------------------
    # GOOGLE CALENDAR SYNC
    # ----------------------------
    try:
        sync_appointment_to_google_calendar(
            appointment_time=appointment_time,
            summary=f"Medical Appointment: {reason or 'Follow-up'}",
            description=f"CareCompanion appointment for patient {patient_id}"
        )
    except Exception as e:
        print("⚠️ Calendar sync failed but DB insert succeeded.")
        print("Error:", e)

    return appointment_id


def get_appointments(user_id: int):
    """Get all appointments for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT id,appointment_time,reason,created_at
        FROM appointments
        WHERE patient_id=%s
        ORDER BY appointment_time;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows

**Notes** Functions

In [ ]:
def add_note(user_id: int, content: str) -> int:
    """Add a note for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        "INSERT INTO notes (user_id,content) VALUES (%s,%s) RETURNING note_id;",
        (user_id,content)
    )
    note_id=cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    return note_id


def get_notes(user_id: int):
    """Get all notes for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT note_id,content,created_at
        FROM notes
        WHERE user_id=%s
        ORDER BY created_at DESC;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows

**Discharge Instructions** Functions

In [ ]:
def add_discharge_instruction(user_id: int, instruction: str) -> int:
    """Add discharge instructions for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        INSERT INTO discharge_instructions (user_id,instruction)
        VALUES (%s,%s)
        RETURNING instruction_id;
        """,
        (user_id,instruction)
    )
    instruction_id=cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    return instruction_id


def get_discharge_instructions(user_id: int):
    """Get discharge instructions for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT instruction_id,instruction,created_at
        FROM discharge_instructions
        WHERE user_id=%s
        ORDER BY created_at DESC;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows

**Symptom Logs** Functions

In [ ]:
def log_symptom(user_id: int, symptom: str, severity: int) -> int:
    """Log a symptom for a user"""
    conn = get_connection()
    cur = conn.cursor()

    # Ensure severity is within bounds
    severity = max(1, min(10, severity))

    cur.execute(
        """
        INSERT INTO symptom_logs (user_id, symptom, severity)
        VALUES (%s, %s, %s)
        RETURNING symptom_id;
        """,
        (user_id, symptom, severity)
    )

    symptom_id = cur.fetchone()[0]
    conn.commit()
    cur.close()
    conn.close()

    return symptom_id

def get_symptom_logs(user_id: int):
    """Get symptom history for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT symptom,severity,logged_at
        FROM symptom_logs
        WHERE user_id=%s
        ORDER BY logged_at DESC;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows


**Medication Logs** Functions

In [ ]:
def add_medication(
    user_id: int,
    medication_name: str,
    dosage: str = None,
    schedule: str = None,
    start_date=None,
    end_date=None
) -> int:
    """Add a medication record for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        INSERT INTO medication_logs
        (user_id,medication_name,dosage,schedule,start_date,end_date)
        VALUES (%s,%s,%s,%s,%s,%s)
        RETURNING medication_id;
        """,
        (user_id,medication_name,dosage,schedule,start_date,end_date)
    )
    medication_id=cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    return medication_id


def get_medications(user_id: int):
    """Get medications for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT medication_name,dosage,schedule,start_date,end_date,created_at
        FROM medication_logs
        WHERE user_id=%s;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows

**Reminders** Functions

In [ ]:
def add_reminder(
    user_id: int,
    reminder_type: str,
    message: str,
    remind_at
) -> int:
    """Add a reminder for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        INSERT INTO reminders (user_id,reminder_type,message,remind_at)
        VALUES (%s,%s,%s,%s)
        RETURNING reminder_id;
        """,
        (user_id,reminder_type,message,remind_at)
    )
    reminder_id=cur.fetchone()[0]

    conn.commit()
    cur.close()
    conn.close()

    return reminder_id


def get_reminders(user_id: int):
    """Get reminders for a user"""
    conn=get_connection()
    cur=conn.cursor()

    cur.execute(
        """
        SELECT reminder_type,message,remind_at,created_at
        FROM reminders
        WHERE user_id=%s
        ORDER BY remind_at;
        """,
        (user_id,)
    )
    rows=cur.fetchall()

    cur.close()
    conn.close()
    return rows

# **Part 3: Test Databases Connection**

In [ ]:
def debug_connection():
    conn=get_db_connection()
    cur=conn.cursor()

    cur.execute("SELECT current_database(), current_user;")
    result=cur.fetchone()

    cur.close()
    conn.close()
    return result


def check_tables():
    conn=get_db_connection()
    cur=conn.cursor()

    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema='public';
    """)

    tables=cur.fetchall()

    cur.close()
    conn.close()
    return tables

In [ ]:
debug_connection()

('neondb', 'neondb_owner')

In [ ]:
check_tables()

[('appointments',),
 ('users',),
 ('patients',),
 ('symptoms',),
 ('medication_logs',),
 ('notes',),
 ('discharge_instructions',),
 ('symptom_logs',),
 ('reminders',)]

# **Part 4: Reference Data** (In-Memory Dictionaries)

Immutable data per patient (utilized as medical reference information)

In [ ]:
MEDICATIONS_DB = {
    # PAIN MEDICATIONS
    "acetaminophen": {
        "purpose": "Relieves mild to moderate pain and reduces fever",
        "common_side_effects": "Generally well-tolerated; rare side effects include nausea",
        "warnings": "Do not exceed 3000mg per day. Do not take with alcohol. Call doctor if pain persists more than 10 days.",
        "food_instructions": "Can be taken with or without food"
    },
    "ibuprofen": {
        "purpose": "Reduces pain, inflammation, and fever",
        "common_side_effects": "Stomach upset, nausea, dizziness",
        "warnings": "Take with food to prevent stomach upset. Do not use if you have kidney problems or stomach ulcers.",
        "food_instructions": "Take with food or milk"
    },
    "oxycodone": {
        "purpose": "Treats moderate to severe pain",
        "common_side_effects": "Drowsiness, constipation, nausea, dizziness",
        "warnings": "Do not drive or operate machinery. Do not drink alcohol. Can be habit-forming.",
        "food_instructions": "Can be taken with or without food"
    },

    # HEART MEDICATIONS
    "metoprolol": {
        "purpose": "Lowers blood pressure and heart rate, protects the heart",
        "common_side_effects": "Tiredness, dizziness, slow heartbeat",
        "warnings": "Do not stop taking suddenly. Call doctor if heart rate drops below 50.",
        "food_instructions": "Take with food"
    },
    "lisinopril": {
        "purpose": "Lowers blood pressure and protects kidneys and heart",
        "common_side_effects": "Dry cough, dizziness, headache",
        "warnings": "Call doctor if you develop swelling of face, lips, or tongue.",
        "food_instructions": "Can be taken with or without food"
    },
    "aspirin": {
        "purpose": "Prevents blood clots, reduces risk of heart attack and stroke",
        "common_side_effects": "Stomach upset, heartburn",
        "warnings": "Can cause bleeding. Call doctor if you see blood in stool or unusual bruising.",
        "food_instructions": "Take with food to reduce stomach upset"
    },
    "atorvastatin": {
        "purpose": "Lowers cholesterol to reduce risk of heart disease",
        "common_side_effects": "Muscle aches, headache, nausea",
        "warnings": "Call doctor if you have unexplained muscle pain or weakness.",
        "food_instructions": "Take in the evening, with or without food"
    },
    "warfarin": {
        "purpose": "Prevents blood clots",
        "common_side_effects": "Bleeding, bruising",
        "warnings": "Requires regular blood tests. Avoid alcohol. Maintain consistent vitamin K intake.",
        "food_instructions": "Take at the same time each day"
    },

    # ANTIBIOTICS
    "amoxicillin": {
        "purpose": "Treats bacterial infections",
        "common_side_effects": "Diarrhea, nausea, rash",
        "warnings": "Complete the full course even if you feel better.",
        "food_instructions": "Can be taken with or without food"
    },
    "ciprofloxacin": {
        "purpose": "Treats bacterial infections",
        "common_side_effects": "Nausea, diarrhea, dizziness",
        "warnings": "Avoid dairy products and antacids within 2 hours. Call doctor if you have tendon pain.",
        "food_instructions": "Take 2 hours before or 6 hours after dairy"
    },

    # DIABETES MEDICATIONS
    "metformin": {
        "purpose": "Lowers blood sugar in type 2 diabetes",
        "common_side_effects": "Nausea, diarrhea, stomach upset",
        "warnings": "Call doctor if you have severe nausea, vomiting, or rapid breathing.",
        "food_instructions": "Take with meals to reduce stomach upset"
    },

    # STOMACH MEDICATIONS
    "omeprazole": {
        "purpose": "Reduces stomach acid, treats heartburn and ulcers",
        "common_side_effects": "Headache, nausea, diarrhea",
        "warnings": "Take 30-60 minutes before eating.",
        "food_instructions": "Take before breakfast"
    },
    "pantoprazole": {
        "purpose": "Reduces stomach acid, protects stomach lining",
        "common_side_effects": "Headache, diarrhea",
        "warnings": "Take 30-60 minutes before a meal.",
        "food_instructions": "Take before breakfast"
    },

    # DIURETICS
    "furosemide": {
        "purpose": "Removes excess fluid, treats swelling and high blood pressure",
        "common_side_effects": "Frequent urination, dizziness, muscle cramps",
        "warnings": "Take in the morning to avoid nighttime urination. Call doctor if severe muscle cramps.",
        "food_instructions": "Can be taken with or without food"
    },

    # STEROIDS
    "prednisone": {
        "purpose": "Reduces inflammation and suppresses immune system",
        "common_side_effects": "Increased appetite, mood changes, trouble sleeping",
        "warnings": "Do not stop suddenly if taking for more than a few days. Take in morning.",
        "food_instructions": "Take with food to prevent stomach upset"
    },

    # BLOOD THINNERS
    "apixaban": {
        "purpose": "Prevents blood clots, reduces stroke risk",
        "common_side_effects": "Bleeding, bruising",
        "warnings": "Call doctor if you have unusual bleeding or blood in urine.",
        "food_instructions": "Can be taken with or without food"
    },
    "clopidogrel": {
        "purpose": "Prevents blood clots after heart attack or stent",
        "common_side_effects": "Bleeding, bruising, stomach upset",
        "warnings": "Do not stop without consulting doctor.",
        "food_instructions": "Can be taken with or without food"
    }
}

print(f"Medications reference database loaded: {len(MEDICATIONS_DB)} medications")


WARNING_SIGNS_DB = {
    "cardiac_surgery": {
        "emergency_signs": [
            "chest pain that is severe or getting worse",
            "difficulty breathing or shortness of breath at rest",
            "sudden weakness on one side of body",
            "fainting or passing out",
            "coughing up blood",
            "heart racing or pounding that does not stop",
            "temperature above 101.5°F with chills"
        ],
        "urgent_signs": [
            "incision is red, swollen, or draining pus",
            "fever above 100.4°F",
            "leg swelling that is new or getting worse",
            "weight gain of more than 3 pounds in one day",
            "nausea or vomiting that prevents taking medications",
            "dizziness that does not go away"
        ],
        "expected_symptoms": [
            "mild soreness around incision",
            "tiredness and fatigue for several weeks",
            "trouble sleeping",
            "mild swelling in legs that improves with elevation",
            "decreased appetite",
            "mood changes or feeling emotional",
            "mild constipation from pain medications"
        ]
    },

    "joint_replacement": {
        "emergency_signs": [
            "sudden severe pain in the surgical leg",
            "chest pain or difficulty breathing",
            "calf pain with swelling and warmth (possible blood clot)",
            "surgical leg turns pale, blue, or cold",
            "fainting or passing out"
        ],
        "urgent_signs": [
            "fever above 100.4°F",
            "increased redness or warmth around incision",
            "drainage from incision that is yellow, green, or smells bad",
            "new numbness or tingling in foot",
            "unable to bear weight as instructed"
        ],
        "expected_symptoms": [
            "pain and swelling around the joint for several weeks",
            "bruising that may spread down the leg",
            "clicking or popping sounds from new joint",
            "difficulty sleeping due to discomfort",
            "stiffness that improves with physical therapy"
        ]
    },

    "abdominal_surgery": {
        "emergency_signs": [
            "severe abdominal pain that is getting worse",
            "vomiting blood or material that looks like coffee grounds",
            "blood in stool or black tarry stools",
            "fever above 101.5°F with chills",
            "unable to keep any fluids down for 12 hours",
            "no bowel movement or gas for 3 days"
        ],
        "urgent_signs": [
            "incision is opening or separating",
            "redness spreading from incision site",
            "fever above 100.4°F",
            "increasing pain not relieved by prescribed medication",
            "persistent nausea or vomiting"
        ],
        "expected_symptoms": [
            "gas pain and bloating for several days",
            "constipation from pain medications",
            "decreased appetite",
            "fatigue",
            "mild bruising around incision"
        ]
    },

    "pneumonia": {
        "emergency_signs": [
            "severe difficulty breathing",
            "chest pain when breathing",
            "confusion or altered mental status",
            "lips or fingernails turning blue",
            "coughing up large amounts of blood"
        ],
        "urgent_signs": [
            "fever that returns after improving",
            "shortness of breath with minimal activity",
            "cough that is getting worse instead of better",
            "unable to keep fluids down"
        ],
        "expected_symptoms": [
            "cough that gradually improves over 2-3 weeks",
            "fatigue for several weeks",
            "mild shortness of breath with activity that improves daily"
        ]
    },

    "heart_failure": {
        "emergency_signs": [
            "severe shortness of breath",
            "chest pain",
            "fainting",
            "coughing up pink or bloody mucus"
        ],
        "urgent_signs": [
            "weight gain of more than 2-3 pounds in one day or 5 pounds in one week",
            "increased swelling in legs, ankles, or abdomen",
            "waking up at night short of breath",
            "needing more pillows to sleep comfortably",
            "new or worsening cough"
        ],
        "expected_symptoms": [
            "some shortness of breath with activity",
            "mild fatigue",
            "need to urinate more often when taking diuretics"
        ]
    },

    "stroke": {
        "emergency_signs": [
            "new weakness or numbness on one side",
            "new difficulty speaking or understanding speech",
            "new vision problems",
            "severe headache unlike any before",
            "new difficulty walking or loss of balance"
        ],
        "urgent_signs": [
            "dizziness that does not go away",
            "confusion or memory problems getting worse",
            "difficulty swallowing"
        ],
        "expected_symptoms": [
            "fatigue, especially in the first few weeks",
            "emotional changes",
            "gradual improvement in strength and coordination with therapy"
        ]
    },

    "general_surgery": {
        "emergency_signs": [
            "severe pain not relieved by medication",
            "heavy bleeding from incision",
            "fever above 101.5°F",
            "difficulty breathing",
            "chest pain"
        ],
        "urgent_signs": [
            "fever above 100.4°F",
            "redness, swelling, or pus from incision",
            "incision opening up",
            "persistent vomiting",
            "unable to urinate"
        ],
        "expected_symptoms": [
            "mild to moderate pain around incision",
            "fatigue",
            "bruising near incision",
            "constipation"
        ]
    }
}

print(f"Warning signs database loaded: {len(WARNING_SIGNS_DB)} conditions")

Medications reference database loaded: 17 medications
Warning signs database loaded: 7 conditions


In [ ]:
# MEDICATIONS LOG DATABASE
# This database stores detailed medication information for patient tracking

MEDICATIONS_LOG_DB = {
    "common_medications": {
        "metoprolol": {
            "generic_name": "Metoprolol",
            "brand_names": ["Lopressor", "Toprol-XL"],
            "drug_class": "Beta-blocker",
            "common_uses": ["high blood pressure", "heart failure", "chest pain", "heart attack prevention"],
            "typical_dosages": ["25mg", "50mg", "100mg", "200mg"],
            "frequency_options": ["once daily", "twice daily"],
            "common_side_effects": ["tiredness", "dizziness", "slow heartbeat", "cold hands/feet"],
            "serious_warnings": ["Do not stop suddenly", "Monitor heart rate", "Report dizziness or fainting"],
            "food_interactions": "Can be taken with or without food",
            "timing_notes": "Take at same time each day"
        },
        "lisinopril": {
            "generic_name": "Lisinopril",
            "brand_names": ["Prinivil", "Zestril"],
            "drug_class": "ACE Inhibitor",
            "common_uses": ["high blood pressure", "heart failure", "post-heart attack"],
            "typical_dosages": ["5mg", "10mg", "20mg", "40mg"],
            "frequency_options": ["once daily"],
            "common_side_effects": ["dry cough", "dizziness", "headache"],
            "serious_warnings": ["Report swelling of face/lips/tongue", "Monitor blood pressure", "Stay hydrated"],
            "food_interactions": "Can be taken with or without food",
            "timing_notes": "Take at same time each day"
        },
        "warfarin": {
            "generic_name": "Warfarin",
            "brand_names": ["Coumadin", "Jantoven"],
            "drug_class": "Anticoagulant (blood thinner)",
            "common_uses": ["blood clot prevention", "atrial fibrillation", "heart valve replacement"],
            "typical_dosages": ["1mg", "2mg", "2.5mg", "5mg", "7.5mg", "10mg"],
            "frequency_options": ["once daily"],
            "common_side_effects": ["easy bruising", "bleeding gums", "nosebleeds"],
            "serious_warnings": ["Requires regular INR monitoring", "Avoid foods high in vitamin K", "Report any unusual bleeding", "Many drug interactions"],
            "food_interactions": "Limit leafy greens, avoid cranberry juice",
            "timing_notes": "Take at same time each evening"
        },
        "furosemide": {
            "generic_name": "Furosemide",
            "brand_names": ["Lasix"],
            "drug_class": "Loop diuretic (water pill)",
            "common_uses": ["fluid retention", "swelling", "high blood pressure", "heart failure"],
            "typical_dosages": ["20mg", "40mg", "80mg"],
            "frequency_options": ["once daily", "twice daily"],
            "common_side_effects": ["increased urination", "dizziness when standing", "low potassium"],
            "serious_warnings": ["Monitor weight daily", "May need potassium supplement", "Report muscle cramps or weakness"],
            "food_interactions": "Take with food if stomach upset",
            "timing_notes": "Take in morning to avoid nighttime urination"
        },
        "aspirin": {
            "generic_name": "Aspirin",
            "brand_names": ["Bayer", "Ecotrin", "Bufferin"],
            "drug_class": "Antiplatelet agent",
            "common_uses": ["heart attack prevention", "stroke prevention", "post-cardiac surgery"],
            "typical_dosages": ["81mg (low-dose)", "325mg"],
            "frequency_options": ["once daily"],
            "common_side_effects": ["stomach upset", "heartburn", "easy bruising"],
            "serious_warnings": ["Take with food", "Report black stools or stomach pain", "Avoid with other blood thinners without doctor approval"],
            "food_interactions": "Take with food or milk",
            "timing_notes": "Can take any time of day, be consistent"
        },
        "atorvastatin": {
            "generic_name": "Atorvastatin",
            "brand_names": ["Lipitor"],
            "drug_class": "Statin (cholesterol medication)",
            "common_uses": ["high cholesterol", "heart disease prevention", "post-heart attack"],
            "typical_dosages": ["10mg", "20mg", "40mg", "80mg"],
            "frequency_options": ["once daily"],
            "common_side_effects": ["muscle aches", "joint pain", "headache"],
            "serious_warnings": ["Report severe muscle pain or weakness", "Avoid grapefruit juice", "Regular liver tests needed"],
            "food_interactions": "Avoid grapefruit and grapefruit juice",
            "timing_notes": "Can take any time of day with or without food"
        },
        "amoxicillin": {
            "generic_name": "Amoxicillin",
            "brand_names": ["Amoxil", "Moxatag"],
            "drug_class": "Antibiotic (penicillin)",
            "common_uses": ["bacterial infections", "pneumonia", "ear infections", "urinary tract infections"],
            "typical_dosages": ["250mg", "500mg", "875mg"],
            "frequency_options": ["twice daily", "three times daily"],
            "common_side_effects": ["diarrhea", "nausea", "rash"],
            "serious_warnings": ["Complete full course even if feeling better", "Report severe diarrhea or rash", "Take with food if stomach upset"],
            "food_interactions": "Can be taken with or without food",
            "timing_notes": "Space doses evenly throughout the day"
        },
        "gabapentin": {
            "generic_name": "Gabapentin",
            "brand_names": ["Neurontin", "Gralise"],
            "drug_class": "Anticonvulsant/Nerve pain medication",
            "common_uses": ["nerve pain", "post-surgical pain", "shingles pain"],
            "typical_dosages": ["100mg", "300mg", "400mg", "600mg", "800mg"],
            "frequency_options": ["three times daily", "once daily at bedtime"],
            "common_side_effects": ["drowsiness", "dizziness", "swelling of feet"],
            "serious_warnings": ["Do not stop suddenly", "May cause drowsiness - avoid driving initially", "Report mood changes"],
            "food_interactions": "Take with food",
            "timing_notes": "Space doses evenly; bedtime dose may help with sleep"
        },
        "oxycodone": {
            "generic_name": "Oxycodone",
            "brand_names": ["OxyContin", "Roxicodone", "Percocet (with acetaminophen)"],
            "drug_class": "Opioid pain medication",
            "common_uses": ["moderate to severe pain", "post-surgical pain"],
            "typical_dosages": ["5mg", "10mg", "15mg", "20mg"],
            "frequency_options": ["every 4-6 hours as needed", "every 12 hours (extended-release)"],
            "common_side_effects": ["constipation", "drowsiness", "nausea", "dizziness"],
            "serious_warnings": ["Risk of addiction", "Take stool softener to prevent constipation", "Do not drink alcohol", "May cause severe drowsiness", "Do not drive"],
            "food_interactions": "Take with food if nausea occurs",
            "timing_notes": "Use only as prescribed; taper gradually when discontinuing"
        },
        "omeprazole": {
            "generic_name": "Omeprazole",
            "brand_names": ["Prilosec", "Losec"],
            "drug_class": "Proton pump inhibitor (PPI)",
            "common_uses": ["acid reflux", "heartburn", "stomach ulcers", "GERD"],
            "typical_dosages": ["20mg", "40mg"],
            "frequency_options": ["once daily"],
            "common_side_effects": ["headache", "nausea", "diarrhea", "stomach pain"],
            "serious_warnings": ["Take 30-60 minutes before first meal", "May decrease magnesium levels with long-term use", "Do not crush or chew capsules"],
            "food_interactions": "Take before breakfast on empty stomach",
            "timing_notes": "Take 30-60 minutes before first meal of the day"
        }
    },

    "medication_schedule_templates": {
        "post_cardiac_surgery": [
            {"name": "metoprolol", "dosage": "50mg", "frequency": "twice daily", "duration": "ongoing"},
            {"name": "aspirin", "dosage": "81mg", "frequency": "once daily", "duration": "ongoing"},
            {"name": "atorvastatin", "dosage": "40mg", "frequency": "once daily at bedtime", "duration": "ongoing"},
            {"name": "oxycodone", "dosage": "5mg", "frequency": "every 4-6 hours as needed", "duration": "2 weeks"}
        ],
        "post_knee_replacement": [
            {"name": "aspirin", "dosage": "325mg", "frequency": "twice daily", "duration": "6 weeks"},
            {"name": "oxycodone", "dosage": "10mg", "frequency": "every 4-6 hours as needed", "duration": "2-3 weeks"},
            {"name": "gabapentin", "dosage": "300mg", "frequency": "three times daily", "duration": "as needed"}
        ],
        "pneumonia_treatment": [
            {"name": "amoxicillin", "dosage": "500mg", "frequency": "three times daily", "duration": "7-10 days"}
        ]
    },

    "medication_reminders_config": {
        "critical_medications": ["warfarin", "insulin", "anticoagulants"],
        "time_sensitive_medications": ["antibiotics", "antihypertensives"],
        "reminder_windows": {
            "strict": "15 minutes",
            "moderate": "1 hour",
            "flexible": "2-3 hours"
        }
    }
}

print(f"Medications Log Database loaded: {len(MEDICATIONS_LOG_DB['common_medications'])} medications")

Medications Log Database loaded: 10 medications


In [ ]:
# REMINDERS DATABASE
# This database manages various types of reminders for patient care

REMINDERS_DB = {
    "reminder_types": {
        "medication": {
            "priority": "high",
            "default_advance_time": "15 minutes",
            "allow_snooze": True,
            "snooze_duration": "15 minutes",
            "max_snoozes": 2,
            "escalation_required": True
        },
        "appointment": {
            "priority": "high",
            "reminder_schedule": ["1 week before", "1 day before", "2 hours before"],
            "allow_snooze": False,
            "preparation_checklist": True
        },
        "symptom_check": {
            "priority": "medium",
            "default_frequency": "daily",
            "preferred_time": "evening",
            "allow_skip": True,
            "skip_limit": "2 consecutive days"
        },
        "wound_care": {
            "priority": "high",
            "default_frequency": "as prescribed",
            "include_instructions": True,
            "photo_prompt": True
        },
        "activity": {
            "priority": "medium",
            "types": ["physical therapy exercises", "walking", "breathing exercises"],
            "allow_reschedule": True,
            "include_instructions": True
        },
        "diet_fluid": {
            "priority": "medium",
            "types": ["fluid intake tracking", "weight monitoring", "dietary restrictions"],
            "default_frequency": "varies by type",
            "allow_skip": False
        },
        "follow_up_prep": {
            "priority": "medium",
            "trigger_time": "2-3 days before appointment",
            "includes": ["symptom summary", "questions list", "medication review", "test results"],
            "auto_generate": True
        },
        "lab_test": {
            "priority": "high",
            "reminder_schedule": ["3 days before", "1 day before"],
            "include_prep_instructions": True,
            "fasting_required_flag": True
        }
    },

    "reminder_templates": {
        "medication_dose": {
            "title": "Time for {medication_name}",
            "message": "Take {dosage} of {medication_name}. {special_instructions}",
            "actions": ["Taken", "Snooze", "Skip (with reason)"]
        },
        "appointment_upcoming": {
            "title": "Appointment Reminder: {provider_name}",
            "message": "You have an appointment with {provider_name} on {date} at {time}. Location: {location}",
            "actions": ["Confirm", "Need to reschedule", "View details"]
        },
        "daily_symptom_check": {
            "title": "Daily Symptom Check-in",
            "message": "How are you feeling today? Let's log your symptoms and track your recovery.",
            "actions": ["Start check-in", "Feeling fine - skip", "Remind me in 2 hours"]
        },
        "wound_care_time": {
            "title": "Wound Care Reminder",
            "message": "Time to {care_type}. {instructions}",
            "actions": ["View instructions", "Completed", "Need help"]
        },
        "exercise_therapy": {
            "title": "Physical Therapy Exercise",
            "message": "Time for your {exercise_type}. {duration} recommended.",
            "actions": ["Start exercises", "Completed", "Reschedule"]
        },
        "weight_monitoring": {
            "title": "Daily Weight Check",
            "message": "Please weigh yourself and log your weight. Report gains of 2+ lbs in one day.",
            "actions": ["Log weight", "View history", "Skip today"]
        },
        "appointment_preparation": {
            "title": "Prepare for Your Appointment",
            "message": "Your appointment with {provider_name} is in {days_until}. Let's prepare your visit summary.",
            "actions": ["Review symptoms", "List questions", "Check medications"]
        },
        "lab_test_prep": {
            "title": "Lab Test Tomorrow",
            "message": "{test_name} scheduled for {time}. {fasting_instructions}",
            "actions": ["View requirements", "Set alarm", "Need to reschedule"]
        }
    },

    "reminder_scheduling_rules": {
        "medication": {
            "once_daily": ["morning", "evening", "bedtime"],
            "twice_daily": ["8am and 8pm", "9am and 9pm"],
            "three_times_daily": ["8am, 2pm, 8pm", "9am, 3pm, 9pm"],
            "four_times_daily": ["8am, 12pm, 4pm, 8pm"],
            "as_needed": "no automatic reminders"
        },
        "symptom_check": {
            "post_surgery_week_1": "twice daily (morning and evening)",
            "post_surgery_week_2_4": "once daily (evening)",
            "chronic_condition": "3 times per week"
        },
        "activity_reminders": {
            "walking": "daily at preferred time",
            "exercises": "per PT schedule (typically 2-3 times daily)",
            "breathing_exercises": "4 times daily for respiratory conditions"
        }
    },

    "notification_preferences": {
        "channels": ["push_notification", "sms", "email", "in_app"],
        "quiet_hours": {
            "default_start": "10:00 PM",
            "default_end": "7:00 AM",
            "exceptions": ["critical_medication", "emergency_symptoms"]
        },
        "urgency_levels": {
            "critical": "immediate notification, override quiet hours",
            "high": "notification within 5 minutes",
            "medium": "notification at next scheduled check",
            "low": "daily summary"
        }
    },

    "smart_reminder_features": {
        "context_aware": {
            "description": "Adjust reminders based on user activity and patterns",
            "examples": [
                "Skip morning medication reminder if already taken (logged)",
                "Suggest earlier exercise time if user typically completes in morning",
                "Increase symptom check frequency if concerning patterns detected"
            ]
        },
        "adherence_tracking": {
            "description": "Monitor completion rates and identify barriers",
            "metrics": ["medication adherence rate", "appointment attendance", "symptom logging frequency"],
            "intervention_threshold": "< 80% adherence for critical items"
        },
        "caregiver_escalation": {
            "description": "Alert caregiver if patient misses critical reminders",
            "triggers": [
                "Missed 2 consecutive medication doses",
                "No symptom check for 48 hours post-surgery",
                "Appointment missed without cancellation"
            ]
        }
    }
}

print(f"Reminders Database loaded: {len(REMINDERS_DB['reminder_types'])} reminder types")

Reminders Database loaded: 8 reminder types


# **Part 4: Creating Custom Agent Tools**

Next tasks to complete:

TOOL 1: **DISCHARGE PARSER** - Amal

TOOL 2: **MEDICATION SCHEDULE + MEDICATION INFO LOOKUP** - Basir

TOOL 3: **SYMPTOM CHECKER** - Samikha

TOOL 4: **VISIT SUMMARY** - Michelle

**Tool 1: Discharge Parser**


In [ ]:
@tool
def parse_discharge_instructions(user_id: int, discharge_text: str) -> str:
    """
    Parse discharge instructions and extract:
    - Medications
    - Appointments
    - Care instructions
    """

    prompt = f"""
You are a medical discharge parser.

Extract structured information from the discharge instructions below.

Return ONLY valid JSON in this format (no markdown fences, no extra text):

{{
    "medications": [
        {{
            "name": "string",
            "dosage": "string",
            "schedule": "string"
        }}
    ],
    "appointments": [
        {{
            "datetime": "string or null",
            "reason": "string"
        }}
    ],
    "care_instructions": [
        "string"
    ]
}}

Discharge Instructions:
{discharge_text}
"""

    try:
        response = _genai_model.generate_content(prompt)
        raw_output = response.text.strip()

        if raw_output.startswith("```"):
            raw_output = raw_output.replace("```json", "").replace("```", "").strip()

        parsed = json.loads(raw_output)

    except Exception as e:
        return f"⚠️ Parsing failed: {str(e)}"

    # Store Medications
    for med in parsed.get("medications", []):
        add_medication(
            user_id=user_id,
            medication_name=med.get("name"),
            dosage=med.get("dosage"),
            schedule=med.get("schedule"),
        )

    # Store Appointments
    for appt in parsed.get("appointments", []):
        dt_string = appt.get("datetime")
        reason = appt.get("reason")

        if dt_string:
            try:
                appointment_time = date_parser.parse(dt_string)
                create_appointment(
                    user_id=user_id,
                    appointment_time=appointment_time,
                    reason=reason,
                )
            except Exception:
                pass

    # Store Care Instructions
    for instruction in parsed.get("care_instructions", []):
        add_discharge_instruction(user_id, instruction)

    return (
        "✅ Discharge instructions processed successfully.\n"
        f"- Medications added: {len(parsed.get('medications', []))}\n"
        f"- Appointments scheduled: {len(parsed.get('appointments', []))}\n"
        f"- Care instructions saved: {len(parsed.get('care_instructions', []))}"
    )

In [ ]:
# Test Discharge Parser Tool

test_text = """
Take Metoprolol 50mg twice daily.
Take Acetaminophen 500mg every 6 hours as needed for pain.
Follow up with your cardiologist on March 20 at 2 PM.
Avoid heavy lifting for 2 weeks.
Monitor for chest pain or shortness of breath.
"""

print(parse_discharge_instructions.invoke({
    "user_id": 1,
    "discharge_text": test_text
}))

✅ Discharge instructions processed successfully.
- Medications added: 2
- Appointments scheduled: 1
- Care instructions saved: 2


**Tool 2: Medication Schedule**

In [ ]:
@tool
def create_medication_schedule(user_id: int) -> str:
    """
    Create a daily medication reminder schedule for a patient.
    Use this tool after discharge info is parsed or when patient asks to see their medication schedule.
    """

    # Get user's medications from database
    medications = get_medications(user_id)

    if not medications:
        return "No medications found for this patient. Please add medications first."

    # Map frequency to times using REMINDERS_DB rules
    frequency_to_times = REMINDERS_DB["reminder_scheduling_rules"]["medication"]

    all_reminders = []
    schedule_output = ["📋 YOUR DAILY MEDICATION SCHEDULE", "=" * 40, ""]

    for med in medications:
        # Handle both tuple and dict formats
        if isinstance(med, tuple):
            med_name = med[0].lower() if med[0] else ""
            dosage = med[1] if len(med) > 1 and med[1] else ""
            frequency = med[2] if len(med) > 2 and med[2] else "once daily"
        else:
            med_name = med.get("medication_name", "").lower()
            dosage = med.get("dosage", "")
            frequency = med.get("schedule", "once daily")

        # Skip if no medication name
        if not med_name:
            continue

        # Get additional info from reference data
        med_info = MEDICATIONS_LOG_DB["common_medications"].get(med_name, {})
        food_instructions = med_info.get("food_interactions", "No specific instructions")
        timing_notes = med_info.get("timing_notes", "")

        # Determine reminder times based on frequency
        freq_lower = frequency.lower() if frequency else "once daily"

        if "once" in freq_lower or "daily" in freq_lower:
            times = ["08:00"]  # 8:00 AM
        elif "twice" in freq_lower:
            times = ["08:00", "20:00"]  # 8 AM and 8 PM
        elif "three" in freq_lower:
            times = ["08:00", "14:00", "20:00"]  # 8 AM, 2 PM, 8 PM
        elif "four" in freq_lower:
            times = ["08:00", "12:00", "16:00", "20:00"]  # 8 AM, 12 PM, 4 PM, 8 PM
        elif "bedtime" in freq_lower:
            times = ["22:00"]  # 10:00 PM
        elif "as needed" in freq_lower or "prn" in freq_lower:
            times = []  # No scheduled reminders for PRN meds
        else:
            times = ["08:00"]  # Default

        # Add to schedule output
        schedule_output.append(f"💊 {med_name.upper()} {dosage}")
        schedule_output.append(f"   Frequency: {frequency}")

        # Format times for display
        display_times = []
        for t in times:
            hour, minute = t.split(':')
            hour_int = int(hour)
            if hour_int < 12:
                display_times.append(f"{hour_int}:{minute} AM")
            elif hour_int == 12:
                display_times.append(f"12:{minute} PM")
            else:
                display_times.append(f"{hour_int-12}:{minute} PM")

        schedule_output.append(f"   Times: {', '.join(display_times) if times else 'As needed'}")
        schedule_output.append(f"   📝 {food_instructions}")
        if timing_notes:
            schedule_output.append(f"   ⏰ {timing_notes}")
        schedule_output.append("")

        # Create reminders in database (using proper timestamp format)
        from datetime import datetime, timedelta

        for time_str in times:
            try:
                # Parse time string (format: "HH:MM")
                hour, minute = map(int, time_str.split(':'))

                # Create reminder time for today at the specified hour
                now = datetime.now()
                reminder_time = now.replace(hour=hour, minute=minute, second=0, microsecond=0)

                # If time already passed today, schedule for tomorrow
                if reminder_time < now:
                    reminder_time = reminder_time + timedelta(days=1)

                add_reminder(
                    user_id=user_id,
                    reminder_type="medication",
                    message=f"Take {dosage} of {med_name}. {food_instructions}",
                    remind_at=reminder_time
                )
                all_reminders.append({"med": med_name, "time": time_str})
            except Exception as e:
                schedule_output.append(f"   ⚠️ Could not create reminder: {str(e)}")

    schedule_output.append("=" * 40)
    schedule_output.append(f"✅ Created {len(all_reminders)} reminders")
    schedule_output.append("⚠️ Set alarms on your phone as backup!")

    return "\n".join(schedule_output)


In [ ]:
# Quick lookup function for medication info
@tool
def get_medication_info(medication_name: str) -> str:
    """
    Look up information about a specific medication including side effects, warnings, and food instructions.
    Use when patient asks about a specific medication.
    """
    med_name = medication_name.lower().strip()
    med_info = MEDICATIONS_LOG_DB["common_medications"].get(med_name)

    if not med_info:
        return f"No information found for '{medication_name}'. Please check the spelling or consult your pharmacist."

    output = [
        f"💊 {med_info.get('generic_name', medication_name).upper()}",
        f"Brand names: {', '.join(med_info.get('brand_names', ['N/A']))}",
        f"Drug class: {med_info.get('drug_class', 'N/A')}",
        "",
        f"📋 Common uses: {', '.join(med_info.get('common_uses', ['N/A']))}",
        f"💊 Typical dosages: {', '.join(med_info.get('typical_dosages', ['N/A']))}",
        "",
        f"⚠️ Side effects: {', '.join(med_info.get('common_side_effects', ['N/A']))}",
        "",
        "🚨 WARNINGS:",
    ]

    for warning in med_info.get("serious_warnings", ["None listed"]):
        output.append(f"   • {warning}")

    output.append("")
    output.append(f"🍽️ Food: {med_info.get('food_interactions', 'No specific instructions')}")
    output.append(f"⏰ Timing: {med_info.get('timing_notes', 'Follow prescription')}")

    return "\n".join(output)

In [ ]:
conn = get_connection()
cur = conn.cursor()
cur.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'medication_logs';
""")
columns = cur.fetchall()
print("medication_logs columns:", [c[0] for c in columns])
cur.close()
conn.close()

medication_logs columns: ['medication_id', 'user_id', 'start_date', 'end_date', 'created_at', 'medication_name', 'dosage', 'schedule']


In [ ]:
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT user_id, name FROM users LIMIT 5;")
users = cur.fetchall()
print("Users:", users)
cur.close()
conn.close()

Users: [(1, 'John Smith'), (2, 'John Smith'), (3, 'John Smith'), (4, 'John Smith'), (5, 'John Smith')]


#### **Adding Test Data**

In [ ]:
# Step 1: Create a test user
conn = get_connection()
cur = conn.cursor()

cur.execute("""
    INSERT INTO users (name, email, phone)
    VALUES ('John Smith', 'john@test.com', '555-1234')
    RETURNING user_id;
""")
user_id = cur.fetchone()[0]
conn.commit()
print(f"Created user with ID: {user_id}")

# Step 2: Add test medications for that user
cur.execute("""
    INSERT INTO medication_logs (user_id, medication_name, dosage, schedule)
    VALUES
        (%s, 'metoprolol', '50mg', 'twice daily'),
        (%s, 'aspirin', '81mg', 'once daily'),
        (%s, 'atorvastatin', '40mg', 'once daily at bedtime');
""", (user_id, user_id, user_id))
conn.commit()
print("Added 3 medications")

cur.close()
conn.close()

Created user with ID: 153
Added 3 medications


In [ ]:
# Test with a user who has medications
print(create_medication_schedule.invoke({"user_id": 1}))

# Test medication lookup
print(get_medication_info.invoke({"medication_name": "metoprolol"}))

📋 YOUR DAILY MEDICATION SCHEDULE

💊 METOPROLOL 50mg
   Frequency: twice daily
   Times: 8:00 AM
   📝 Can be taken with or without food
   ⏰ Take at same time each day

💊 ASPIRIN 81mg
   Frequency: once daily
   Times: 8:00 AM
   📝 Take with food or milk
   ⏰ Can take any time of day, be consistent

💊 ATORVASTATIN 40mg
   Frequency: once daily at bedtime
   Times: 8:00 AM
   📝 Avoid grapefruit and grapefruit juice
   ⏰ Can take any time of day with or without food

💊 METOPROLOL 50mg
   Frequency: twice daily
   Times: 8:00 AM
   📝 Can be taken with or without food
   ⏰ Take at same time each day

💊 ACETAMINOPHEN 500mg
   Frequency: every 6 hours as needed for pain
   Times: As needed
   📝 No specific instructions

💊 METOPROLOL 50mg
   Frequency: twice daily
   Times: 8:00 AM
   📝 Can be taken with or without food
   ⏰ Take at same time each day

💊 ACETAMINOPHEN 500mg
   Frequency: every 6 hours as needed for pain
   Times: As needed
   📝 No specific instructions

💊 METOPROLOL 50mg
   Fr

**Tool 3: Symptom Checker**

In [ ]:
@tool
def log_symptom_check(user_id: int, symptom: str, severity: int, condition_type: Optional[str] = None) -> str:
    """
    Log a symptom for a patient and check if it requires medical attention.
    Use this when patient reports a symptom or when doing daily symptom check-ins.

    Args:
        user_id: The patient's user ID
        symptom: Description of the symptom
        severity: Severity level (1-10, where 1 is mild and 10 is severe)
        condition_type: Optional condition type (e.g., 'cardiac_surgery', 'joint_replacement',
                       'pneumonia', 'heart_failure', 'stroke', 'abdominal_surgery', 'general_surgery')
    """

    # Log the symptom to database
    try:
        symptom_id = log_symptom(user_id, symptom, severity)
    except Exception as e:
        return f"❌ Error logging symptom: {str(e)}"

    # Get recent symptom history
    recent_symptoms = get_symptom_logs(user_id)

    # Prepare response
    response = [
        f"✅ Symptom logged: '{symptom}' (Severity: {severity}/10)",
        f"   Symptom ID: {symptom_id}",
        ""
    ]

    # Check for warning signs if condition type is provided
    if condition_type and condition_type.lower() in WARNING_SIGNS_DB:
        condition_data = WARNING_SIGNS_DB[condition_type.lower()]
        warning_assessment = assess_symptom_severity(symptom.lower(), severity, condition_data)

        if warning_assessment["level"] == "emergency":
            response.extend([
                "🚨 **EMERGENCY WARNING** 🚨",
                "⚠️ Your symptom requires IMMEDIATE medical attention:",
                f"   • {warning_assessment['reason']}",
                "",
                "**IMMEDIATE ACTIONS:**",
                "1. Call 911 or go to the nearest emergency room NOW",
                "2. Do not wait - this is urgent",
                "3. If possible, have someone drive you",
                "4. Take your medication list with you",
                ""
            ])
        elif warning_assessment["level"] == "urgent":
            response.extend([
                "⚠️ **URGENT CARE NEEDED** ⚠️",
                "You should contact your doctor TODAY:",
                f"   • {warning_assessment['reason']}",
                "",
                "**RECOMMENDED ACTIONS:**",
                "1. Call your doctor's office immediately",
                "2. Describe these symptoms to the nurse/doctor",
                "3. Ask if you need to be seen today",
                "4. Follow their guidance about medications",
                ""
            ])
        elif warning_assessment["level"] == "monitor":
            response.extend([
                "📊 **MONITOR THIS SYMPTOM**",
                f"   • {warning_assessment['reason']}",
                "",
                "**WHAT TO DO:**",
                "1. Monitor if symptom gets worse",
                "2. Rest and follow your discharge instructions",
                "3. Check again in 4-6 hours",
                "4. Contact doctor if it persists or worsens",
                ""
            ])
        else:
            # Expected/normal symptom
            expected_info = condition_data.get("expected_symptoms", [])
            matching_expected = [s for s in expected_info if any(word in symptom.lower() for word in s.lower().split())]

            if matching_expected:
                response.extend([
                    "✅ **EXPECTED SYMPTOM**",
                    f"   This is a normal part of recovery: {matching_expected[0]}",
                    "",
                    "**RECOVERY TIPS:**",
                    "1. Continue following your discharge plan",
                    "2. Rest when needed",
                    "3. Stay hydrated",
                    "4. Let us know if it becomes more severe",
                    ""
                ])
            else:
                response.extend([
                    "ℹ️ **SYMPTOM LOGGED**",
                    "   This symptom will be tracked. Continue monitoring and",
                    "   contact your doctor if you have concerns.",
                    ""
                ])

    # Provide context from recent symptoms
    if len(recent_symptoms) > 1:
        response.append("📈 **Recent Symptom History:**")
        for i, (past_symptom, past_severity, past_time) in enumerate(recent_symptoms[:3]):
            time_str = past_time.strftime("%b %d, %I:%M %p") if hasattr(past_time, 'strftime') else str(past_time)
            response.append(f"   • {past_symptom} (Severity: {past_severity}/10) - {time_str}")

        # Check for worsening trends
        if len(recent_symptoms) >= 3:
            recent_severities = [s[1] for s in recent_symptoms[:3] if s[0].lower() == symptom.lower()]
            if len(recent_severities) >= 2 and recent_severities[0] > recent_severities[1]:
                response.extend([
                    "",
                    "⚠️ **TREND ALERT:** This symptom appears to be getting worse.",
                    "   Please monitor closely and contact your doctor if this continues."
                ])
        response.append("")

    # Add self-care guidance
    response.extend([
        "🏠 **Self-Care Reminders:**",
        "• Take medications as prescribed",
        "• Stay hydrated (unless fluid restricted)",
        "• Get adequate rest",
        "• Follow your discharge instructions",
        "• Contact your care team with concerns",
        "",
        "📞 **Emergency:** Call 911 for severe symptoms",
        "📞 **Doctor's Office:** Call during business hours for non-emergency concerns"
    ])

    return "\n".join(response)

In [ ]:
@tool
def daily_symptom_checkin(user_id: int, condition_type: Optional[str] = None) -> str:
    """
    Perform a daily symptom check-in for a patient.
    Asks about common post-discharge symptoms and logs them.
    Use this for routine daily monitoring.

    Args:
        user_id: The patient's user ID
        condition_type: Optional condition type for targeted questions
    """

    # Get today's date
    today = datetime.now().date()

    # Get recent symptom logs
    recent_symptoms = get_symptom_logs(user_id)

    # Check if already checked in today
    checked_today = False
    if recent_symptoms:
        latest_time = recent_symptoms[0][2]  # logged_at timestamp
        if hasattr(latest_time, 'date') and latest_time.date() == today:
            checked_today = True

    response = [
        "🌅 **DAILY SYMPTOM CHECK-IN**",
        "=" * 40,
        "",
    ]

    if checked_today:
        response.extend([
            "📝 You've already completed a check-in today.",
            "Here's your most recent symptom log:",
            ""
        ])
        for symptom, severity, log_time in recent_symptoms[:3]:
            time_str = log_time.strftime("%I:%M %p") if hasattr(log_time, 'strftime') else str(log_time)
            response.append(f"   • {symptom} (Severity: {severity}/10) - {time_str}")
        response.append("")

    # Generate condition-specific questions if condition_type provided
    if condition_type and condition_type.lower() in WARNING_SIGNS_DB:
        condition_data = WARNING_SIGNS_DB[condition_type.lower()]

        response.extend([
            f"**Based on your condition ({condition_type.replace('_', ' ').title()}):**",
            "",
            "**Common symptoms to monitor:**"
        ])

        for symptom in condition_data.get("expected_symptoms", [])[:4]:
            response.append(f"   • {symptom}")

        response.extend([
            "",
            "**WATCH FOR:**"
        ])

        for warning in condition_data.get("urgent_signs", [])[:3]:
            response.append(f"   ⚠️ {warning}")

        response.extend([
            "",
            "**EMERGENCY WARNING SIGNS:**"
        ])

        for emergency in condition_data.get("emergency_signs", [])[:3]:
            response.append(f"   🚨 {emergency}")

        response.append("")

    # General questions
    response.extend([
        "**Today's Check-In Questions:**",
        "1. How is your pain level today? (1-10)",
        "2. Any new or worsening symptoms?",
        "3. Have you taken all medications as prescribed?",
        "4. How is your energy level?",
        "5. Any concerns about your recovery?",
        "",
        "💡 **To log a symptom, use the log_symptom_check tool**",
        "   Example: 'Log symptom: headache, severity: 4'",
        "",
        "📊 **Recovery Tip:**",
        "• Track symptoms daily to notice patterns",
        "• Report any sudden changes to your doctor",
        "• Bring your symptom log to follow-up appointments",
        "• Don't hesitate to ask questions"
    ])

    return "\n".join(response)

In [ ]:
@tool
def analyze_symptom_trends(user_id: int, days: int = 7) -> str:
    """
    Analyze symptom trends for a patient over a specified time period.
    Use this to identify patterns or improvements in recovery.

    Args:
        user_id: The patient's user ID
        days: Number of days to analyze (default: 7)
    """

    # Get symptom history
    all_symptoms = get_symptom_logs(user_id)

    if not all_symptoms:
        return "No symptom data available. Start logging symptoms to track trends!"

    # Filter by date range
    from datetime import timedelta
    cutoff_date = datetime.now() - timedelta(days=days)

    recent_symptoms = []
    for symptom, severity, log_time in all_symptoms:
        if hasattr(log_time, 'date') and log_time >= cutoff_date:
            recent_symptoms.append((symptom, severity, log_time))
        elif isinstance(log_time, str):
            # Handle string timestamps if needed
            recent_symptoms.append((symptom, severity, log_time))

    if not recent_symptoms:
        return f"No symptoms logged in the last {days} days."

    # Organize by symptom type
    symptom_data: Dict[str, List[int]] = {}
    symptom_timeline = []

    for symptom, severity, log_time in recent_symptoms:
        if symptom not in symptom_data:
            symptom_data[symptom] = []
        symptom_data[symptom].append(severity)

        time_str = log_time.strftime("%b %d") if hasattr(log_time, 'strftime') else str(log_time)
        symptom_timeline.append(f"   • {time_str}: {symptom} ({severity}/10)")

    response = [
        f"📊 **SYMPTOM TREND ANALYSIS (Last {days} Days)**",
        "=" * 50,
        "",
        f"**Total symptoms logged:** {len(recent_symptoms)}",
        f"**Unique symptoms:** {len(symptom_data)}",
        ""
    ]

    # Analyze each symptom
    response.append("**Symptom Breakdown:**")
    for symptom, severities in symptom_data.items():
        avg_severity = sum(severities) / len(severities)
        max_severity = max(severities)
        trend = "improving" if len(severities) > 1 and severities[-1] < severities[0] else \
                "worsening" if len(severities) > 1 and severities[-1] > severities[0] else \
                "stable"

        response.extend([
            f"   • {symptom}:",
            f"     - Occurrences: {len(severities)}",
            f"     - Average severity: {avg_severity:.1f}/10",
            f"     - Peak severity: {max_severity}/10",
            f"     - Trend: {trend} 📈" if trend == "worsening" else f"     - Trend: {trend} 📉" if trend == "improving" else f"     - Trend: {trend} ⚖️"
        ])

    response.extend([
        "",
        "**Symptom Timeline (Most Recent):**"
    ])

    # Show 10 most recent logs
    for log in symptom_timeline[:10]:
        response.append(log)

    if len(symptom_timeline) > 10:
        response.append(f"   ... and {len(symptom_timeline) - 10} more")

    response.extend([
        "",
        "**Observations:**"
    ])

    # Add insights
    worsening_symptoms = [s for s, sev in symptom_data.items() if len(sev) > 1 and sev[-1] > sev[0]]
    improving_symptoms = [s for s, sev in symptom_data.items() if len(sev) > 1 and sev[-1] < sev[0]]

    if worsening_symptoms:
        response.append(f"   ⚠️ Symptoms worsening: {', '.join(worsening_symptoms)}")
    if improving_symptoms:
        response.append(f"   ✅ Symptoms improving: {', '.join(improving_symptoms)}")

    # Check for severe symptoms
    severe_symptoms = [f"{s} ({max(sev)}/10)" for s, sev in symptom_data.items() if max(sev) >= 7]
    if severe_symptoms:
        response.extend([
            "",
            "🚨 **ALERT:** Severe symptoms detected (7+ severity)",
            f"   • {', '.join(severe_symptoms)}",
            "   Please contact your healthcare provider to discuss these symptoms."
        ])

    return "\n".join(response)

In [ ]:
def assess_symptom_severity(symptom: str, severity: int, condition_data: dict) -> dict:
    """
    Helper function to assess symptom severity based on condition data.
    Returns assessment level and reason.
    """
    symptom_lower = symptom.lower()

    # Check emergency signs (severity high or matches emergency list)
    emergency_signs = condition_data.get("emergency_signs", [])
    for sign in emergency_signs:
        if any(word in symptom_lower for word in sign.lower().split()):
            if severity >= 7:
                return {"level": "emergency", "reason": f"This matches: {sign}"}

    # Check urgent signs
    urgent_signs = condition_data.get("urgent_signs", [])
    for sign in urgent_signs:
        if any(word in symptom_lower for word in sign.lower().split()):
            if severity >= 5:
                return {"level": "urgent", "reason": f"This could indicate: {sign}"}
            else:
                return {"level": "monitor", "reason": f"This may be related to: {sign}. Monitor closely."}

    # Check if it's a severe symptom even if not in lists
    if severity >= 8:
        return {"level": "urgent", "reason": f"Severe symptom (severity {severity}/10) requires medical attention."}

    if severity >= 5:
        return {"level": "monitor", "reason": f"Moderate symptom (severity {severity}/10). Monitor and rest."}

    return {"level": "normal", "reason": "Mild symptom within expected range."}


# Export the tools
symptom_checker_tools = [
    log_symptom_check,
    daily_symptom_checkin,
    analyze_symptom_trends
]

**Testing Symptom Checker Tool**

In [ ]:
# Comprehensive Test for Symptom Checker Tool

import time
from datetime import datetime, timedelta

print("=" * 60)
print("🧪 SYMPTOM CHECKER TOOL - DATABASE INTEGRATION TESTS")
print("=" * 60)

# PART 1: SETUP - Establish Database Connection
print("\n📋 PART 1: Connecting to Database")
print("-" * 40)

# Test database connection first
try:
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT current_database(), current_user;")
    db_info = cur.fetchone()
    print(f"✅ Connected to database: {db_info[0]} as user: {db_info[1]}")
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials and try again.")
    exit()

# Create a dedicated test user in the actual database
print("\n📋 Creating Test User in Database")
print("-" * 40)

conn = get_db_connection()
cur = conn.cursor()

# Check if test user already exists
cur.execute("SELECT user_id FROM users WHERE email = 'symptom.test@example.com'")
existing = cur.fetchone()

if existing:
    test_user_id = existing[0]
    print(f"✅ Found existing test user with ID: {test_user_id}")

    # Clear old test symptoms to start fresh
    cur.execute("DELETE FROM symptom_logs WHERE user_id = %s", (test_user_id,))
    conn.commit()
    print(f"🧹 Cleared old symptom logs for test user")
else:
    # Create new test user
    cur.execute("""
        INSERT INTO users (name, email, phone)
        VALUES ('Symptom Test Patient', 'symptom.test@example.com', '555-TEST')
        RETURNING user_id;
    """)
    test_user_id = cur.fetchone()[0]
    conn.commit()
    print(f"✅ Created new test user with ID: {test_user_id}")

cur.close()
conn.close()

# PART 2: Test Basic Symptom Logging (with database verification)
print("\n📋 PART 2: Testing Basic Symptom Logging")
print("-" * 40)

def verify_symptom_in_db(user_id, symptom_text, severity):
    """Helper function to verify symptom was actually saved to database"""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT symptom, severity, logged_at
        FROM symptom_logs
        WHERE user_id = %s AND symptom = %s AND severity = %s
        ORDER BY logged_at DESC LIMIT 1;
    """, (user_id, symptom_text, severity))
    result = cur.fetchone()
    cur.close()
    conn.close()
    return result is not None

# Test: Log a mild symptom
print("\n🔹 Test 2.1: Logging mild headache")
result = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "Mild headache",
    "severity": 3
})
print(result)

# Verify in database
if verify_symptom_in_db(test_user_id, "Mild headache", 3):
    print("✅ DATABASE VERIFIED: Symptom successfully stored")
else:
    print("❌ DATABASE ERROR: Symptom not found in database")
assert verify_symptom_in_db(test_user_id, "Mild headache", 3), "❌ Failed: Symptom not in database"

# Test: Log a moderate symptom
print("\n🔹 Test 2.2: Logging moderate fatigue")
result = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "Feeling tired",
    "severity": 5
})
print(result)

if verify_symptom_in_db(test_user_id, "Feeling tired", 5):
    print("✅ DATABASE VERIFIED: Symptom successfully stored")
else:
    print("❌ DATABASE ERROR: Symptom not found in database")


# PART 3: Test Database Retrieval Functions
print("\n📋 PART 3: Testing Database Retrieval")
print("-" * 40)

# Test get_symptom_logs function
print("\n🔹 Test 3.1: Retrieving symptoms from database")
retrieved_symptoms = get_symptom_logs(test_user_id)
print(f"Retrieved {len(retrieved_symptoms)} symptoms from database:")
for symptom, severity, logged_at in retrieved_symptoms[:5]:  # Show first 5
    time_str = logged_at.strftime("%Y-%m-%d %H:%M") if hasattr(logged_at, 'strftime') else str(logged_at)
    print(f"   • {symptom} (Severity: {severity}) - {time_str}")

assert len(retrieved_symptoms) >= 2, "❌ Failed: Should have at least 2 symptoms in database"
print("✅ Database retrieval working correctly")

# PART 4: Test Condition-Specific Warning Signs
print("\n📋 PART 4: Testing Condition-Specific Warning Signs")
print("-" * 40)

# Test: Cardiac surgery - emergency sign
print("\n🔹 Test 4.1: Cardiac surgery - emergency sign")
result = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "Severe chest pain radiating to arm",
    "severity": 9,
    "condition_type": "cardiac_surgery"
})
print(result)

# Verify it was saved
if verify_symptom_in_db(test_user_id, "Severe chest pain radiating to arm", 9):
    print("✅ DATABASE VERIFIED: Emergency symptom stored")

# PART 5: Test Multiple Symptoms for Trend Analysis
print("\n📋 PART 5: Creating Data for Trend Analysis")
print("-" * 40)

print("\n🔹 Creating symptom history for headache...")
conn = get_db_connection()
cur = conn.cursor()
base_time = datetime.now() - timedelta(days=5)

severity_values = [1, 2, 3, 4, 5]

for i, severity in enumerate(severity_values):
    symptom_time = base_time + timedelta(days=i)
    try:
        cur.execute("""
            INSERT INTO symptom_logs (user_id, symptom, severity, logged_at)
            VALUES (%s, %s, %s, %s);
        """, (test_user_id, "Headache", severity, symptom_time))
        print(f"  ✅ Added headache severity {severity}")
    except Exception as e:
        print(f"  ❌ Failed to add severity {severity}: {e}")
        break

conn.commit()
cur.close()
conn.close()
print(f"✅ Added 5 headache symptoms with increasing severity")


# PART 6: Test Trend Analysis with Database Data
print("\n📋 PART 6: Testing Symptom Trend Analysis")
print("-" * 40)

# Test: Analyze trends from database
print("\n🔹 Test 6.1: Analyzing 7-day trend")
result = analyze_symptom_trends.invoke({
    "user_id": test_user_id,
    "days": 7
})
print(result)

# Test: Verify trend detection by querying database directly
print("\n🔹 Test 6.2: Direct database trend verification")
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT symptom, severity, logged_at
    FROM symptom_logs
    WHERE user_id = %s AND symptom = 'Headache'
    ORDER BY logged_at;
""", (test_user_id,))
headache_data = cur.fetchall()
cur.close()
conn.close()

if len(headache_data) >= 2:
    first_sev = headache_data[0][1]
    last_sev = headache_data[-1][1]
    print(f"Headache trend: Started at {first_sev}/10, now at {last_sev}/10")
    if last_sev > first_sev:
        print("✅ DATABASE VERIFIED: Worsening trend detected correctly")
    else:
        print("❌ Database shows incorrect trend")


# PART 7: Test Daily Check-in with Database Context
print("\n📋 PART 7: Testing Daily Check-in with Database History")
print("-" * 40)

print("\n🔹 Test 7.1: Daily check-in showing recent history")
result = daily_symptom_checkin.invoke({
    "user_id": test_user_id,
    "condition_type": "cardiac_surgery"
})
print(result)

# Verify check-in references database data
if "You've already completed a check-in today" in result or "Recent Symptom History" in result:
    print("✅ Daily check-in is pulling data from database")
else:
    print("⚠️ Check-in may not be showing database history")


# PART 8: Test Database Integrity
print("\n📋 PART 8: Testing Database Integrity")
print("-" * 40)

# Count total symptoms for test user
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM symptom_logs WHERE user_id = %s", (test_user_id,))
total_symptoms = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n📊 DATABASE SUMMARY:")
print(f"   • Total symptoms logged for test user: {total_symptoms}")
print(f"   • Unique symptom types: Should include headache, chest pain, fatigue, etc.")

# Verify foreign key integrity
conn = get_db_connection()
cur = conn.cursor()
try:
    cur.execute("""
        SELECT COUNT(*)
        FROM symptom_logs s
        LEFT JOIN users u ON s.user_id = u.user_id
        WHERE u.user_id IS NULL;
    """)
    orphaned = cur.fetchone()[0]
    if orphaned == 0:
        print("✅ Foreign key integrity: No orphaned records")
    else:
        print(f"❌ Found {orphaned} orphaned symptom records")
except Exception as e:
    print(f"Could not verify foreign keys: {e}")
finally:
    cur.close()
    conn.close()


# PART 9: Final Verification
print("\n" + "=" * 60)
print("📊 FINAL DATABASE VERIFICATION")
print("=" * 60)

# Get final count and sample
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT symptom, severity, logged_at
    FROM symptom_logs
    WHERE user_id = %s
    ORDER BY logged_at DESC
    LIMIT 5;
""", (test_user_id,))
recent = cur.fetchall()

cur.execute("SELECT COUNT(*) FROM symptom_logs WHERE user_id = %s", (test_user_id,))
final_count = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n✅ DATABASE VERIFICATION PASSED:")
print(f"   • User ID {test_user_id} has {final_count} symptoms in database")
print(f"\n📝 Most recent 5 symptoms (from database):")
for symptom, severity, logged_at in recent:
    time_str = logged_at.strftime("%Y-%m-%d %H:%M") if hasattr(logged_at, 'strftime') else str(logged_at)
    print(f"   • {symptom} (Severity: {severity}) - {time_str}")

🧪 SYMPTOM CHECKER TOOL - DATABASE INTEGRATION TESTS

📋 PART 1: Connecting to Database
----------------------------------------
✅ Connected to database: neondb as user: neondb_owner

📋 Creating Test User in Database
----------------------------------------
✅ Found existing test user with ID: 15
🧹 Cleared old symptom logs for test user

📋 PART 2: Testing Basic Symptom Logging
----------------------------------------

🔹 Test 2.1: Logging mild headache
✅ Symptom logged: 'Mild headache' (Severity: 3/10)
   Symptom ID: 542

🏠 **Self-Care Reminders:**
• Take medications as prescribed
• Stay hydrated (unless fluid restricted)
• Get adequate rest
• Follow your discharge instructions
• Contact your care team with concerns

📞 **Emergency:** Call 911 for severe symptoms
📞 **Doctor's Office:** Call during business hours for non-emergency concerns
✅ DATABASE VERIFIED: Symptom successfully stored

🔹 Test 2.2: Logging moderate fatigue
✅ Symptom logged: 'Feeling tired' (Severity: 5/10)
   Symptom ID: 5

**Tool 4: Visit Summary**

In [ ]:
@tool
def generate_visit_summary(user_id: int) -> str:
    """
    Fetches symptom logs and medications for a user and generates a
    concise summary for a doctor's visit.
    """
    symptoms = get_symptom_logs(user_id)
    medications = get_medications(user_id)
    appointments = get_appointments(user_id)

    # Format symptoms with dates
    symptom_str = ""
    if symptoms:
        for s in symptoms[:10]:  # Last 10 symptoms
            if hasattr(s[2], 'strftime'):
                date_str = s[2].strftime("%b %d, %Y")
            else:
                date_str = str(s[2])
            symptom_str += f"- {s[0]} (Severity: {s[1]}/10) on {date_str}\n"
    else:
        symptom_str = "No symptoms logged."

    # Format medications
    med_str = ""
    if medications:
        for m in medications:
            if isinstance(m, tuple):
                med_str += f"- {m[0]} {m[1]}: {m[2]}\n"
            else:
                med_str += f"- {m.get('medication_name', 'Unknown')} {m.get('dosage', '')}: {m.get('schedule', '')}\n"
    else:
        med_str = "No medications listed."

    # Format appointments
    appt_str = ""
    if appointments:
        for a in appointments:
            if hasattr(a[1], 'strftime'):
                appt_time = a[1].strftime("%b %d, %Y at %I:%M %p")
            else:
                appt_time = str(a[1])
            appt_str += f"- {a[2] or 'Follow-up'} on {appt_time}\n"
    else:
        appt_str = "No upcoming appointments."

    prompt_text = f"""
    You are a medical assistant helping a patient prepare for a doctor's visit.
    Based on the following data, provide a concise, well-organized summary.

    SYMPTOMS (Recent):
    {symptom_str}

    CURRENT MEDICATIONS:
    {med_str}

    UPCOMING APPOINTMENTS:
    {appt_str}

    Please format your response as follows:

    📊 **VISIT SUMMARY**

    **Overall Health Trend:**
    [Brief assessment - are symptoms improving/worsening/stable?]

    **Key Concerns to Discuss:**
    • [List 2-3 main symptoms or issues]

    **Medication Adherence:**
    [Brief note on medications]

    **Suggested Questions for Your Doctor:**
    1. [Question 1 based on symptoms]
    2. [Question 2 based on medications]
    3. [Question 3 about next steps]

    **What to Bring:**
    • This summary
    • Current medication list
    • Insurance card
    • List of questions
    """

    try:
        response = _genai_model.generate_content(prompt_text)
        return f"VISIT SUMMARY\n{'-'*40}\n{response.text}"
    except Exception as e:
        return f"❌ Error generating summary: {str(e)}"


**Helper Functions**

In [ ]:
def assess_symptom_severity(symptom: str, severity: int, condition_data: dict) -> dict:
    """
    Helper function to assess symptom severity based on condition data.
    Returns assessment level and reason.
    """
    symptom_lower = symptom.lower()

    # Check emergency signs (severity high or matches emergency list)
    emergency_signs = condition_data.get("emergency_signs", [])
    for sign in emergency_signs:
        if any(word in symptom_lower for word in sign.lower().split()):
            if severity >= 7:
                return {"level": "emergency", "reason": f"This matches: {sign}"}

    # Check urgent signs
    urgent_signs = condition_data.get("urgent_signs", [])
    for sign in urgent_signs:
        if any(word in symptom_lower for word in sign.lower().split()):
            if severity >= 5:
                return {"level": "urgent", "reason": f"This could indicate: {sign}"}
            else:
                return {"level": "monitor", "reason": f"This may be related to: {sign}. Monitor closely."}

    # Check if it's a severe symptom even if not in lists
    if severity >= 8:
        return {"level": "urgent", "reason": f"Severe symptom (severity {severity}/10) requires medical attention."}

    if severity >= 5:
        return {"level": "monitor", "reason": f"Moderate symptom (severity {severity}/10). Monitor and rest."}

    return {"level": "normal", "reason": "Mild symptom within expected range."}

**DATABASE FIX - Update symptom_logs constraint**

In [ ]:
def fix_database_constraints():
    """
    Run this once to fix the symptom severity constraint
    """
    try:
        conn = get_connection()
        cur = conn.cursor()

        # Check current constraint
        cur.execute("""
            SELECT conname
            FROM pg_constraint
            WHERE conrelid = 'symptom_logs'::regclass
            AND contype = 'c';
        """)
        constraints = cur.fetchall()

        if constraints:
            print("Found constraints:", constraints)
            # Drop the constraint if it exists
            for constraint in constraints:
                try:
                    cur.execute(f'ALTER TABLE symptom_logs DROP CONSTRAINT {constraint[0]};')
                    print(f"Dropped constraint: {constraint[0]}")
                except:
                    pass

        # Add new constraint (1-10)
        try:
            cur.execute("""
                ALTER TABLE symptom_logs
                ADD CONSTRAINT symptom_logs_severity_check
                CHECK (severity >= 1 AND severity <= 10);
            """)
            print("Added new severity constraint (1-10)")
        except Exception as e:
            print(f"Note: {e}")

        conn.commit()
        cur.close()
        conn.close()
        print("✅ Database constraints fixed")
    except Exception as e:
        print(f"⚠️ Could not fix constraints: {e}")

**ENHANCED PDF RAG TOOL (with file upload handling)**

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        return f"Error extracting PDF: {str(e)}"


def consult_discharge_instructions(query: str, pdf_text: Optional[str] = None) -> str:
    """
    Searches the uploaded discharge instructions to answer specific
    questions about the patient's recovery, diet, or follow-up care.

    Args:
        query: The user's question
        pdf_text: Optional pre-extracted PDF text
    """
    try:
        # If PDF text is provided directly, use it
        if pdf_text:
            text = pdf_text
        else:
            # Fallback to default file
            reader = PdfReader("discharge_summary.pdf")
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"

        # Use Gemini to answer the query based on the PDF text
        prompt = f"""
        Based on the following discharge instructions, please answer the patient's question.

        DISCHARGE INSTRUCTIONS:
        {text[:3000]}  # Limit to first 3000 chars to avoid token limits

        PATIENT'S QUESTION:
        {query}

        Please provide a clear, helpful answer based ONLY on the information in the discharge instructions.
        If the answer cannot be found in the instructions, politely say so and suggest they contact their doctor.
        """

        response = _genai_model.generate_content(prompt)
        return response.text

    except FileNotFoundError:
        return "No discharge PDF found. Please upload your discharge instructions first."
    except Exception as e:
        return f"Error processing PDF: {str(e)}"


def medical_knowledge_retrieval(topic: str) -> str:
    """
    Accesses the internal medical database for drug purposes,
    side effects, and surgical warning signs.
    """
    topic = topic.lower().strip()

    # Check medications database first
    if topic in MEDICATIONS_DB:
        info = MEDICATIONS_DB[topic]
        return f"💊 {topic.title()}:\n" + "\n".join([f"   • {k}: {v}" for k, v in info.items()])

    # Check warning signs database
    for condition, data in WARNING_SIGNS_DB.items():
        if topic in condition.replace('_', ' '):
            result = [f"⚠️ {condition.replace('_', ' ').title()} Warning Signs:\n"]
            result.append("🚨 EMERGENCY:")
            for sign in data.get('emergency_signs', [])[:3]:
                result.append(f"   • {sign}")
            result.append("\n⚠️ URGENT:")
            for sign in data.get('urgent_signs', [])[:3]:
                result.append(f"   • {sign}")
            return "\n".join(result)

    # Search for partial matches
    matches = []
    for k, v in MEDICATIONS_DB.items():
        if topic in k or topic in str(v).lower():
            matches.append(f"💊 {k.title()}: {v.get('purpose', '')[:100]}...")

    for condition, data in WARNING_SIGNS_DB.items():
        if topic in condition.replace('_', ' '):
            matches.append(f"⚠️ {condition.replace('_', ' ').title()}: See warning signs")

    if matches:
        return "Found these related topics:\n" + "\n".join(matches[:3])

    return "No specific medical reference found for that topic. Please consult your healthcare provider."

**Testing Every Tool** (Patient: Maria Garcia)

In [ ]:
# Comprehensive Tool Testing for Part 4
print("=" * 60)
print("🧪 COMPREHENSIVE TOOL TESTING - PART 4")
print("=" * 60)

# 1. Create test patient
print("\n📋 STEP 1: Creating Test Patient")
print("-" * 40)
test_user_id = create_user("Maria Garcia", "maria@test.com", "555-9999")
print(f"✅ Created user: Maria Garcia (ID: {test_user_id})")

# 2. Parse discharge instructions
print("\n📋 STEP 2: Testing Discharge Parser Tool")
print("-" * 40)
discharge_text = """
Patient discharged after cardiac surgery.

MEDICATIONS:
- Metoprolol 50mg twice daily
- Aspirin 81mg once daily
- Atorvastatin 40mg at bedtime

FOLLOW-UP:
- Cardiologist on April 1, 2026 at 2 PM
- Primary care in 2 weeks

INSTRUCTIONS:
- No lifting over 10 pounds for 6 weeks
- Monitor for chest pain or shortness of breath
- Weigh yourself daily
"""

result = parse_discharge_instructions.invoke({
    "user_id": test_user_id,
    "discharge_text": discharge_text
})
print(result)

# Verify database entries were created
conn = get_connection()
cur = conn.cursor()

# Check medications
cur.execute("SELECT COUNT(*) FROM medication_logs WHERE user_id = %s", (test_user_id,))
med_count = cur.fetchone()[0]
print(f"\n📊 Database verification: {med_count} medications added")

# Check appointments
cur.execute("SELECT COUNT(*) FROM appointments WHERE patient_id = %s", (test_user_id,))
appt_count = cur.fetchone()[0]
print(f"📊 Database verification: {appt_count} appointments added")

# Check discharge instructions
cur.execute("SELECT COUNT(*) FROM discharge_instructions WHERE user_id = %s", (test_user_id,))
inst_count = cur.fetchone()[0]
print(f"📊 Database verification: {inst_count} care instructions added")

cur.close()
conn.close()

# 3. Create medication schedule
print("\n📋 STEP 3: Testing Medication Schedule Tool")
print("-" * 40)
schedule_result = create_medication_schedule.invoke({"user_id": test_user_id})
print(schedule_result)

# Verify reminders were created
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM reminders WHERE user_id = %s", (test_user_id,))
reminder_count = cur.fetchone()[0]
cur.close()
conn.close()
print(f"\n📊 Database verification: {reminder_count} reminders created")

# 4. Test Medication Info Lookup
print("\n📋 STEP 4: Testing Medication Info Tool")
print("-" * 40)
print("\n🔹 Looking up Metoprolol:")
print(get_medication_info.invoke({"medication_name": "metoprolol"}))

print("\n🔹 Looking up Aspirin:")
print(get_medication_info.invoke({"medication_name": "aspirin"}))

# 5. Log symptoms (multiple to test trend analysis)
print("\n📋 STEP 5: Testing Symptom Checker Tool")
print("-" * 40)

# Log first symptom
print("\n🔹 Logging symptom #1:")
result1 = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "mild chest discomfort",
    "severity": 4,
    "condition_type": "cardiac_surgery"
})
print(result1)

# Log second symptom (different)
print("\n🔹 Logging symptom #2:")
result2 = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "fatigue",
    "severity": 6,
    "condition_type": "cardiac_surgery"
})
print(result2)

# Log third symptom (worsening trend)
print("\n🔹 Logging symptom #3 (worsening):")
result3 = log_symptom_check.invoke({
    "user_id": test_user_id,
    "symptom": "chest discomfort",
    "severity": 7,
    "condition_type": "cardiac_surgery"
})
print(result3)

# 6. Test Daily Symptom Check-in
print("\n📋 STEP 6: Testing Daily Symptom Check-in Tool")
print("-" * 40)
checkin_result = daily_symptom_checkin.invoke({
    "user_id": test_user_id,
    "condition_type": "cardiac_surgery"
})
print(checkin_result)

# 7. Test Symptom Trend Analysis
print("\n📋 STEP 7: Testing Symptom Trend Analysis Tool")
print("-" * 40)
trend_result = analyze_symptom_trends.invoke({
    "user_id": test_user_id,
    "days": 7
})
print(trend_result)

# 8. Generate Visit Summary
print("\n📋 STEP 8: Testing Visit Summary Tool")
print("-" * 40)
summary_result = generate_visit_summary.invoke({"user_id": test_user_id})
print(summary_result)

# 9. Test Medical Knowledge Retrieval
print("\n📋 STEP 9: Testing Medical Knowledge Retrieval")
print("-" * 40)
print("\n🔹 Query: 'What are warning signs after cardiac surgery?'")
print(medical_knowledge_retrieval("cardiac surgery"))

print("\n🔹 Query: 'Tell me about warfarin'")
print(medical_knowledge_retrieval("warfarin"))

# 10. Final Database Verification
print("\n📋 STEP 10: Final Database Verification")
print("-" * 40)

conn = get_connection()
cur = conn.cursor()

# Get all data for this user
cur.execute("""
    SELECT
        (SELECT COUNT(*) FROM medication_logs WHERE user_id = %s) as medications,
        (SELECT COUNT(*) FROM appointments WHERE patient_id = %s) as appointments,
        (SELECT COUNT(*) FROM discharge_instructions WHERE user_id = %s) as instructions,
        (SELECT COUNT(*) FROM symptom_logs WHERE user_id = %s) as symptoms,
        (SELECT COUNT(*) FROM reminders WHERE user_id = %s) as reminders
""", (test_user_id, test_user_id, test_user_id, test_user_id, test_user_id))

stats = cur.fetchone()
cur.close()
conn.close()

print("\n📊 FINAL DATABASE STATISTICS:")
print(f"   • Medications: {stats[0]}")
print(f"   • Appointments: {stats[1]}")
print(f"   • Instructions: {stats[2]}")
print(f"   • Symptoms logged: {stats[3]}")
print(f"   • Reminders created: {stats[4]}")

print("\n" + "=" * 60)
print("✅ ALL TOOLS TESTED SUCCESSFULLY!")
print("=" * 60)
print("\n🎯 Summary:")
print("   ✓ Discharge Parser - Extracts medications, appointments, instructions")
print("   ✓ Medication Schedule - Creates daily schedule with reminders")
print("   ✓ Medication Info - Provides detailed medication information")
print("   ✓ Symptom Checker - Logs symptoms with severity assessment")
print("   ✓ Daily Check-in - Guides daily symptom monitoring")
print("   ✓ Trend Analysis - Identifies symptom patterns")
print("   ✓ Visit Summary - Generates doctor-ready summaries")
print("   ✓ Medical Knowledge - Retrieves reference information")
print("=" * 60)

🧪 COMPREHENSIVE TOOL TESTING - PART 4

📋 STEP 1: Creating Test Patient
----------------------------------------
Created user: Maria Garcia (ID: 154)
✅ Created user: Maria Garcia (ID: 154)

📋 STEP 2: Testing Discharge Parser Tool
----------------------------------------
✅ Discharge instructions processed successfully.
- Medications added: 3
- Appointments scheduled: 2
- Care instructions saved: 3

📊 Database verification: 3 medications added
📊 Database verification: 0 appointments added
📊 Database verification: 3 care instructions added

📋 STEP 3: Testing Medication Schedule Tool
----------------------------------------
📋 YOUR DAILY MEDICATION SCHEDULE

💊 METOPROLOL 50mg
   Frequency: twice daily
   Times: 8:00 AM
   📝 Can be taken with or without food
   ⏰ Take at same time each day

💊 ASPIRIN 81mg
   Frequency: once daily
   Times: 8:00 AM
   📝 Take with food or milk
   ⏰ Can take any time of day, be consistent

💊 ATORVASTATIN 40mg
   Frequency: at bedtime
   Times: 10:00 PM
   📝 Avoi

# **Part 5: Implementing Langchain Tools**

**Google Calendar API Integration**
Implemented above.

In [ ]:
# Test Google Calendar API Integration to create a sample event
create_appointment(
    patient_id=1,
    appointment_time=datetime.now(ZoneInfo("America/New_York")),
    reason="Test"
)

⚠️ Calendar sync failed but DB insert succeeded.
Error: [Errno 2] No such file or directory: 'service_account.json'


41

# **Part 6: RAG**

1. **The PDF RAG Tool**: allows users to upload a PDF (like a hospital discharge summary) and let the agent parse it to answer questions

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

def consult_discharge_instructions(query: str) -> str:
    """
    Searches the uploaded discharge instructions to answer specific
    questions about the patient's recovery, diet, or follow-up care.
    """
    try:
        # Assumes you have a file named 'discharge_summary.pdf' in your Colab files
        reader = PdfReader("discharge_summary.pdf")
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        return text
    except Exception as e:
        return "No discharge PDF found. Please upload 'discharge_summary.pdf' to the files sidebar."

2. **The Medical Knowledge RAG Tool**: Retrieves information from the MEDICATIONS_DB dictionary so that the agent can provide proper information, even if the user misspells words in their queries.

In [ ]:
def medical_knowledge_retrieval(topic: str) -> str:
    """
    Accesses the internal medical database for drug purposes,
    side effects, and surgical warning signs.
    """
    topic = topic.lower().strip()
    # Basic retrieval logic from MEDICATIONS_DB
    if topic in MEDICATIONS_DB:
        return str(MEDICATIONS_DB[topic])

    # Search for partial matches if exact match fails
    matches = [f"{k}: {v}" for k, v in MEDICATIONS_DB.items() if topic in k or topic in str(v).lower()]
    return "\n".join(matches[:2]) if matches else "No specific medical reference found for that topic."

**Testing RAG Tools**

In [ ]:
# Test the PDF RAG tool directly
print("--- Testing PDF Extraction ---")
query = "What are my restrictions?"
result = consult_discharge_instructions(query)

if "No discharge PDF found" in result:
    print("❌ FAILED: The function can't find the file. Ensure it is named 'discharge_summary.pdf' in the /content folder.")
else:
    print("✅ SUCCESS! Here is a snippet of the text found:")
    print("-" * 30)
    print(result[:500] + "...") # Prints the first 500 characters

--- Testing PDF Extraction ---
❌ FAILED: The function can't find the file. Ensure it is named 'discharge_summary.pdf' in the /content folder.


In [ ]:
async def ask(prompt: str) -> str:
    final_text = ""
    try:
        # We pass the message to the runner
        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=prompt)]
            )
        ):
            if event.is_final_response():
                final_text = event.response.candidates[0].content.parts[0].text
    except Exception as e:
        return f"Error: {e}"
    return final_text

In [ ]:
from google.genai import types

# ── Don't pre-create — let the runner auto-create on first run ────────────────
USER_ID    = "patient_001"
SESSION_ID = "session_001"   # any fixed string — runner creates it on demand

async def ask(prompt: str) -> str:
    final_text = ""
    try:
        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=prompt)]
            )
        ):
            if event.is_final_response():
                final_text = event.response.candidates[0].content.parts[0].text
    except Exception as e:
        return f"Error: {e}"
    return final_text

# ── Test ──────────────────────────────────────────────────────────────────────
print(await ask("Can I lift my 20lb cat?"))

Error: name 'runner' is not defined


In [ ]:
import google.adk
print(f"ADK Version: {google.adk.__version__}")

ADK Version: 1.26.0


# **Part 7: Agent Integration, Testing**

Next tasks to complete:

1. **Agent Integration**

2. **Conversation Flow Testing**

3. **Streamlit (Final UI Polish & Demo Script)**

1. Agent Integration

In [ ]:
care_companion = Agent(
    name="CareCompanion",
    model=model,
    instruction="""
    You are CareCompanion, a supportive healthcare assistant.
    Your goal is to help patients manage post-discharge care.
    - Use 'medical_knowledge_retrieval' for medication and general symptom questions.
    - Use 'consult_discharge_instructions' for specific recovery plan questions.
    - Use 'create_appointment' to schedule follow-ups.
    - Use 'log_symptom' to record how the patient is feeling.
    Always be empathetic, clear, and concise.
    The current patient is James Whitfield (user ID: 1). Always use user_id=1 for all tool
  calls that require a user_id. Never ask the patient for their user ID.
    """,
    tools=[
        create_appointment,
        get_appointments,
        log_symptom,
        get_symptom_logs,
        add_note,
        get_discharge_instructions,
        consult_discharge_instructions,
        medical_knowledge_retrieval
    ]
)


from google.genai import types

USER_ID    = "patient_001"   # identifies the patient across sessions
SESSION_ID = "session_001"   # any fixed string; runner creates the session on demand

runner = InMemoryRunner(agent=care_companion)
await runner.session_service.create_session(
      app_name=runner.app_name,
      user_id=USER_ID,
      session_id=SESSION_ID
  )
print("✓ Agent, Runner, and Session ready!")

async def ask(prompt: str) -> str:
    final_text = ""
    try:
        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=prompt)]
            )
        ):
            if hasattr(event, "is_final_response") and event.is_final_response():
                if hasattr(event, "content") and event.content:
                    for part in event.content.parts:
                      if hasattr(part, "text") and part.text:
                            final_text += part.text
    except Exception as e:
        return f"Error: {e}"
    return final_text.strip() if final_text else "Agent is thinking or calling a tool..."

print("✓ Agent and Runner integrated successfully!")

✓ Agent, Runner, and Session ready!
✓ Agent and Runner integrated successfully!
